# Prompt Management System Example

This notebook demonstrates how to use the PromptManager class for managing prompts in your LLM applications.


In [ ]:
import os
import json
import mlflow
from pathlib import Path

from llm_ops_pipeline.utils.prompt_management import PromptManager

# Configure MLflow for tracking
os.environ["MLFLOW_TRACKING_URI"] = "./mlruns"

# Initialize the prompt manager
prompt_manager = PromptManager(
    langfuse_api_key=os.environ.get("LANGFUSE_API_KEY"),
    langfuse_secret_key=os.environ.get("LANGFUSE_SECRET_KEY"),
    environment="development"
)

## 1. Creating Prompts

Let's create some example prompts for different use cases.

In [ ]:
# 1. Sentiment Analysis Prompt
sentiment_prompt = prompt_manager.create_prompt(
    prompt_id="classification/sentiment",
    content="""Analyze the sentiment of the following text and classify it as POSITIVE, NEGATIVE, or NEUTRAL.
Only respond with the sentiment label.

Text: {{text}}
Sentiment:""",
    description="A simple sentiment analysis prompt",
    tags=["classification", "sentiment", "simple"]
)

print("Created sentiment prompt:")
print(json.dumps(sentiment_prompt, indent=2))

In [ ]:
# 2. Few-shot classification prompt
few_shot_prompt = prompt_manager.create_prompt(
    prompt_id="classification/few_shot",
    content="""Classify the text into one of the following categories: Technology, Business, Entertainment, Sports, Politics.

Here are some examples:
{{#each examples}}
Text: {{this.text}}
Category: {{this.category}}
{{/each}}

Now, classify the following text:
Text: {{text}}
Category:""",
    description="Few-shot classification prompt with examples",
    tags=["classification", "few-shot"]
)

In [ ]:
# 3. Chain-of-thought reasoning prompt
cot_prompt = prompt_manager.create_prompt(
    prompt_id="reasoning/cot",
    content="""Solve the following problem step-by-step, showing your reasoning:

Problem: {{problem}}

Step-by-step solution:""",
    description="Chain of thought prompt for reasoning problems",
    tags=["reasoning", "cot"]
)

## 2. Retrieving Prompts

Now let's retrieve the prompts we created and use them.

In [ ]:
# Get the sentiment prompt
sentiment = prompt_manager.get_prompt("classification/sentiment")
print(f"Retrieved sentiment prompt version: {sentiment.get('metadata', {}).get('version')}")
print(sentiment['content'])

In [ ]:
# List all available prompts
prompts = prompt_manager.list_prompts()
print(f"Found {len(prompts)} prompts:")
for prompt in prompts:
    print(f"- {prompt['id']} (v{prompt['version']})")

## 3. Using Prompts with OpenAI

Let's use the prompts with OpenAI's API as an example.

In [ ]:
import openai
from string import Template

# Set up OpenAI API key
openai.api_key = os.environ.get("OPENAI_API_KEY")

def run_prompt(prompt_id, inputs):
    """Run a prompt with OpenAI."""
    # Get the prompt
    prompt_data = prompt_manager.get_prompt(prompt_id)
    prompt_template = prompt_data["content"]
    
    # Simple template substitution
    template = Template(prompt_template)
    final_prompt = template.safe_substitute(**inputs)
    
    # Call OpenAI
    response = openai.ChatCompletion.create(
        model="gpt-4-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."}, 
            {"role": "user", "content": final_prompt}
        ],
        max_tokens=100
    )
    
    completion = response.choices[0].message.content
    
    # Log prompt usage to both Langfuse and MLflow
    prompt_manager.log_prompt_usage(
        prompt_id=prompt_id,
        inputs=inputs,
        completion=completion,
        metadata={
            "metrics": {
                "tokens_used": response.usage.total_tokens,
                "latency": response.response_ms / 1000 if hasattr(response, "response_ms") else None
            },
            "model": "gpt-4-turbo"
        }
    )
    
    return completion

In [ ]:
# Test the sentiment analysis prompt
with mlflow.start_run(run_name="sentiment_analysis_test"):
    mlflow.log_param("test_type", "sentiment")
    
    # Analyze multiple texts
    texts = [
        "I really enjoyed the movie. The actors were great and the plot was engaging.",
        "The service was terrible and the food was cold.",
        "The weather today is cloudy with a chance of rain."
    ]
    
    for i, text in enumerate(texts):
        sentiment = run_prompt("classification/sentiment", {"text": text})
        print(f"Text: {text}\nSentiment: {sentiment}\n")

## 4. Using the Decorator Pattern

The PromptManager provides a decorator for easy prompt usage.

In [ ]:
# Define a function using the decorator
@prompt_manager.with_prompt("reasoning/cot")
def solve_problem(prompt, problem_text):
    # The prompt is automatically injected by the decorator
    prompt_template = Template(prompt["content"])
    final_prompt = prompt_template.safe_substitute(problem=problem_text)
    
    # Call OpenAI
    response = openai.ChatCompletion.create(
        model="gpt-4-turbo",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."}, 
            {"role": "user", "content": final_prompt}
        ],
        max_tokens=300
    )
    
    completion = response.choices[0].message.content
    
    # Log usage
    prompt_manager.log_prompt_usage(
        prompt_id="reasoning/cot",
        inputs={"problem": problem_text},
        completion=completion
    )
    
    return completion

In [ ]:
# Test the chain-of-thought function
with mlflow.start_run(run_name="cot_reasoning_test"):
    problem = "If John has 5 apples and Mary has twice as many apples as John, how many apples do they have in total?"
    solution = solve_problem(problem)
    print(solution)

## 5. Updating Prompts

Let's update one of our prompts and see the version change.

In [ ]:
# Update the sentiment prompt with better instructions
updated_sentiment = prompt_manager.update_prompt(
    prompt_id="classification/sentiment",
    content="""Analyze the sentiment of the following text and classify it as POSITIVE, NEGATIVE, or NEUTRAL.
Respond with only the sentiment label - no explanations or additional text.

Guidelines:
- POSITIVE: Text expresses happiness, satisfaction, approval, or positive emotions.
- NEGATIVE: Text expresses sadness, dissatisfaction, criticism, or negative emotions.
- NEUTRAL: Text states facts or contains no clear positive or negative sentiment.

Text: {{text}}
Sentiment:""",
    description="An improved sentiment analysis prompt with clear guidelines",
    tags=["classification", "sentiment", "improved"]
)

print(f"Updated sentiment prompt to version: {updated_sentiment['metadata']['version']}")

In [ ]:
# Test the updated prompt
with mlflow.start_run(run_name="updated_sentiment_test"):
    # Test the same texts
    texts = [
        "I really enjoyed the movie. The actors were great and the plot was engaging.",
        "The service was terrible and the food was cold.",
        "The weather today is cloudy with a chance of rain."
    ]
    
    for i, text in enumerate(texts):
        sentiment = run_prompt("classification/sentiment", {"text": text})
        print(f"Text: {text}\nSentiment: {sentiment}\n")

## 6. Few-Shot Learning with External Dataset

Let's create a small dataset for few-shot learning and use it with our prompt.

In [ ]:
# Create datasets directory if it doesn't exist
datasets_dir = Path(prompt_manager.repo_root) / "prompts" / "datasets"
datasets_dir.mkdir(exist_ok=True)

# Create a simple dataset for text classification
categories_dir = datasets_dir / "categories"
categories_dir.mkdir(exist_ok=True)

examples = [
    {
        "text": "Apple released a new iPhone with advanced AI capabilities.",
        "category": "Technology"
    },
    {
        "text": "The stock market reached a new all-time high yesterday.",
        "category": "Business"
    },
    {
        "text": "The latest Marvel movie broke box office records on opening weekend.",
        "category": "Entertainment"
    },
    {
        "text": "The local team won the championship after an overtime goal.",
        "category": "Sports"
    },
    {
        "text": "Congress passed a new bill addressing climate change initiatives.",
        "category": "Politics"
    }
]

with open(categories_dir / "examples.json", "w") as f:
    json.dump(examples, f, indent=2)

print(f"Created dataset at {categories_dir / 'examples.json'}")

In [ ]:
# Test the few-shot prompt with our dataset
with mlflow.start_run(run_name="few_shot_classification_test"):
    # Load the examples dataset
    with open(categories_dir / "examples.json", "r") as f:
        examples_data = json.load(f)
    
    # Get our prompt
    prompt_data = prompt_manager.get_prompt("classification/few_shot")
    prompt_template = prompt_data["content"]
    
    # For Handlebars-style templates, we need a more sophisticated template engine
    # Here we'll use a simple function to replace the examples section
    def format_examples(template, examples, text):
        examples_text = ""
        for ex in examples:
            examples_text += f"Text: {ex['text']}\nCategory: {ex['category']}\n\n"
        
        # Replace the examples section
        processed_template = template.replace(
            "{{#each examples}}\nText: {{this.text}}\nCategory: {{this.category}}\n{{/each}}", 
            examples_text.strip()
        )
        
        # Replace the input text
        final_prompt = processed_template.replace("{{text}}", text)
        return final_prompt
    
    # Test texts
    test_texts = [
        "The new quantum computing startup secured $50 million in funding.",
        "The president met with foreign leaders to discuss trade agreements.",
        "The concert was sold out within minutes of tickets going on sale."
    ]
    
    for text in test_texts:
        # Format the prompt with examples
        final_prompt = format_examples(prompt_template, examples_data, text)
        
        # Call OpenAI
        response = openai.ChatCompletion.create(
            model="gpt-4-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."}, 
                {"role": "user", "content": final_prompt}
            ],
            max_tokens=50
        )
        
        completion = response.choices[0].message.content
        
        # Log usage
        prompt_manager.log_prompt_usage(
            prompt_id="classification/few_shot",
            inputs={"text": text, "examples": examples_data},
            completion=completion
        )
        
        print(f"Text: {text}\nCategory: {completion}\n")

## 7. Analyzing Prompt Performance with MLflow

Now that we've logged various prompt usages, let's see how to analyze their performance.

In [ ]:
# Query MLflow for our experiments
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiments = client.search_experiments()

print("Available experiments:")
for exp in experiments:
    print(f"- {exp.name} (ID: {exp.experiment_id})")
    
# Get all runs for the default experiment
experiment_id = "0"
runs = client.search_runs(experiment_id)

print(f"\nFound {len(runs)} runs in the default experiment:")
for run in runs:
    print(f"- {run.info.run_id}: {run.data.tags.get('mlflow.runName')}")
    prompt_id = run.data.params.get("prompt_id")
    prompt_version = run.data.params.get("prompt_version")
    if prompt_id and prompt_version:
        print(f"  Prompt: {prompt_id} (v{prompt_version})")
    if "tokens_used" in run.data.metrics:
        print(f"  Tokens used: {run.data.metrics['tokens_used']}")
    if "latency" in run.data.metrics:
        print(f"  Latency: {run.data.metrics['latency']:.3f} seconds")
    print()

In [ ]:
# Get average metrics by prompt version
print("Performance by prompt version:")
sentiment_runs = [
    r for r in runs 
    if r.data.params.get("prompt_id") == "classification/sentiment"
]

# Group by version
by_version = {}
for run in sentiment_runs:
    version = run.data.params.get("prompt_version")
    if version not in by_version:
        by_version[version] = {
            "runs": 0,
            "tokens": [],
            "latency": []
        }
    
    by_version[version]["runs"] += 1
    if "tokens_used" in run.data.metrics:
        by_version[version]["tokens"].append(run.data.metrics["tokens_used"])
    if "latency" in run.data.metrics:
        by_version[version]["latency"].append(run.data.metrics["latency"])

# Calculate averages
for version, data in by_version.items():
    avg_tokens = sum(data["tokens"]) / len(data["tokens"]) if data["tokens"] else "N/A"
    avg_latency = sum(data["latency"]) / len(data["latency"]) if data["latency"] else "N/A"
    
    print(f"\nSentiment Classification v{version} ({data['runs']} runs):")
    print(f"  Average tokens: {avg_tokens:.2f}" if isinstance(avg_tokens, float) else f"  Average tokens: {avg_tokens}")
    print(f"  Average latency: {avg_latency:.3f}s" if isinstance(avg_latency, float) else f"  Average latency: {avg_latency}")

## 8. Conclusion

In this notebook, we've seen how to:

1. Create and manage prompts with the PromptManager
2. Use prompts with LLM APIs
3. Update prompts and track versions
4. Use the decorator pattern for cleaner code
5. Work with few-shot learning datasets
6. Log and analyze prompt performance

The PromptManager provides a robust system for prompt versioning, collaborative engineering, and performance tracking, integrating with Langfuse, MLflow, and DVC for a complete MLOps solution.